In [1]:
import os
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings

In [2]:
# 15 documents: the first 3 deep learning docs are intentionally near-identical —
# all describe gradient descent as the core optimisation technique, with minor phrasing variation.
# The next 3 deep learning docs cover distinct regularisation/training concepts.
# At lambda_mult=1.0 (pure relevance), a query about deep learning training returns
# all 3 near-identical gradient descent docs, showing the redundancy problem.
# As lambda_mult decreases, MMR penalises already-selected similar docs and
# picks one gradient descent doc + Dropout + Batch Norm + LR Scheduler instead.
docs = [
    Document(page_content="Training a deep learning model involves iteratively adjusting weights using gradient descent to minimise the loss.", metadata={"topic": "deep learning"}),
    Document(page_content="Deep learning models are optimised through gradient descent, which updates weights in the direction that reduces the training loss.", metadata={"topic": "deep learning"}),
    Document(page_content="Gradient descent is the core optimisation technique in deep learning, guiding weight updates based on computed gradients of the loss.", metadata={"topic": "deep learning"}),
    Document(page_content="Dropout randomly disables a fraction of neurons during training to prevent overfitting in deep networks.", metadata={"topic": "deep learning"}),
    Document(page_content="Batch normalisation stabilises training by normalising layer inputs, which allows the use of higher learning rates.", metadata={"topic": "deep learning"}),
    Document(page_content="Learning rate schedulers dynamically adjust the learning rate during training to improve convergence and avoid overshooting.", metadata={"topic": "deep learning"}),
    Document(page_content="Arctic sea ice has declined by about 13% per decade since satellite measurements began in 1979.", metadata={"topic": "climate"}),
    Document(page_content="Carbon capture technology removes CO2 from the atmosphere and stores it underground.", metadata={"topic": "climate"}),
    Document(page_content="The permafrost in Siberia contains vast amounts of methane that could be released as it thaws.", metadata={"topic": "climate"}),
    Document(page_content="The Renaissance was a cultural movement in Europe from the 14th to 17th century that revived classical art.", metadata={"topic": "art"}),
    Document(page_content="Impressionism emerged in 19th-century France, focusing on light, colour, and everyday subjects.", metadata={"topic": "art"}),
    Document(page_content="Abstract expressionism prioritises spontaneous, automatic, and subconscious creation.", metadata={"topic": "art"}),
    Document(page_content="Common law systems derive legal principles from judicial precedent rather than written codes.", metadata={"topic": "law"}),
    Document(page_content="The presumption of innocence requires the prosecution to prove guilt beyond reasonable doubt.", metadata={"topic": "law"}),
    Document(page_content="Intellectual property law protects creations of the mind, including patents, trademarks, and copyrights.", metadata={"topic": "law"}),
]

In [3]:
# Embed documents and store in ChromaDB
embeddings = OllamaEmbeddings(model="qwen3-embedding:latest")
vectorstore = Chroma.from_documents(documents=docs, embedding=embeddings, persist_directory="mmr_chroma_db")

In [4]:
query = "deep learning model training and its optimization techniques"

In [11]:
sim_search_results = vectorstore.as_retriever( search_type="mmr",
    search_kwargs={"k": 3,"lambda_mult": 0.1} )

In [12]:
results = sim_search_results.invoke(query)
results

[Document(id='5ae8cba4-a3bd-49b0-a957-2e91b408b472', metadata={'topic': 'deep learning'}, page_content='Training a deep learning model involves iteratively adjusting weights using gradient descent to minimise the loss.'),
 Document(id='74007ed3-bf3d-4495-9fd6-6f5e2dbc3fbd', metadata={'topic': 'art'}, page_content='Abstract expressionism prioritises spontaneous, automatic, and subconscious creation.'),
 Document(id='672202a6-a6d1-48aa-88fc-59b67bd064b3', metadata={'topic': 'climate'}, page_content='Arctic sea ice has declined by about 13% per decade since satellite measurements began in 1979.')]

In [13]:
lambda_values = [1.0, 0.7, 0.5 ,0.0]
for lm in lambda_values:
    sim_search_results = vectorstore.as_retriever( search_type="mmr",
        search_kwargs={"k": 3,"lambda_mult": lm} )
    results = sim_search_results.invoke(query)
    print(f"Lambda Mult: {lm}")
    for i, doc in enumerate(results):
        print(f"Result {i+1}: {doc.page_content} (Topic: {doc.metadata['topic']})")
    print("\n")

Lambda Mult: 1.0
Result 1: Training a deep learning model involves iteratively adjusting weights using gradient descent to minimise the loss. (Topic: deep learning)
Result 2: Training a deep learning model involves iteratively adjusting weights using gradient descent to minimise the loss. (Topic: deep learning)
Result 3: Deep learning models are optimised through gradient descent, which updates weights in the direction that reduces the training loss. (Topic: deep learning)


Lambda Mult: 0.7
Result 1: Training a deep learning model involves iteratively adjusting weights using gradient descent to minimise the loss. (Topic: deep learning)
Result 2: Dropout randomly disables a fraction of neurons during training to prevent overfitting in deep networks. (Topic: deep learning)
Result 3: Learning rate schedulers dynamically adjust the learning rate during training to improve convergence and avoid overshooting. (Topic: deep learning)


Lambda Mult: 0.5
Result 1: Training a deep learning model